# ASQA Baseline — Does Retrieved Context Help?

Sprint-1 relevance baseline for ASQA.

Current completed retrievers:

- DPR
- BM25
- Contriever

ColBERT will be added to this same notebook when its full-corpus retrieval
and generation are complete.

## What This Notebook Shows

This notebook evaluates the integrity and structure of the completed ASQA
Sprint-1 generation matrix.

The scientific comparison is:

**WITHOUT_CONTEXT vs relevance-only retrieved Top-5 context**

for each of the three completed retrievers and all three frozen LLMs.

The primary ASQA answer-correctness metric is official **Disambig-F1 /
QA-F1**. That scorer is not approximated here. Correctness results will be
added only after the frozen official scorer implementation and provenance
are complete.

In [ ]:
from pathlib import Path
import json

import pandas as pd

REPO = Path.cwd().resolve()
if REPO.name == "notebooks":
    REPO = REPO.parent

WS = REPO.parent

GEN_ROOT = (
    WS
    / "data/asqa/generation_outputs_2026-09-09/sprint1"
)

assert GEN_ROOT.is_dir(), GEN_ROOT

print("repo:", REPO)
print("generation root:", GEN_ROOT)

## Dataset — ASQA

Canonical protected evaluation set:

- ASQA official dev split
- 948 questions
- evidence role: `PROJECT_PROTECTED_FINAL`

Canonical retrieval corpus:

- DPR Wikipedia snapshot lineage: 2018-12-20
- 21,015,324 passages
- exact DPR passage body is the canonical context surface
- no corpus subsampling

Each WITH_CONTEXT generation receives exactly **5 passages**.

## Experimental Design

Sprint 1 is a relevance-only baseline.

No MMR, KMeans, Agglomerative clustering, or DPP is used here.

Current matrix:

- WITHOUT_CONTEXT × 3 LLMs
- DPR Top-5 × 3 LLMs
- BM25 Top-5 × 3 LLMs
- Contriever Top-5 × 3 LLMs

Expected generations:

`948 × 12 = 11,376`

ColBERT will add another:

`948 × 3 = 2,844`

when available.

In [ ]:
design_table = pd.DataFrame(
    [
        ["WITHOUT_CONTEXT", "-", 0, "Complete"],
        ["WITH_CONTEXT", "DPR", 5, "Complete"],
        ["WITH_CONTEXT", "BM25", 5, "Complete"],
        ["WITH_CONTEXT", "Contriever", 5, "Complete"],
        ["WITH_CONTEXT", "ColBERT", 5, "Pending"],
    ],
    columns=[
        "Condition",
        "Retriever",
        "Passages to maKI",
        "Sprint-1 status",
    ],
)

design_table

## Generation Settings

Frozen ASQA generation settings:

- temperature = 0
- n = 1
- max_tokens = 512
- exactly five passages for WITH_CONTEXT
- no titles, scores, qrels, document IDs, or retriever identity exposed in
  the prompt context

Physical maKI models:

1. llama-3.3-70b
2. gemma4-26b
3. qwen3.6-36b

The historical logical model directory `ministral-3-14b` maps to the
physical qwen3.6-36b endpoint.

## Generation Integrity

In [ ]:
records = []

for path in sorted(GEN_ROOT.rglob("sample_*.json")):
    payload = json.loads(path.read_text(encoding="utf-8"))

    parsed = payload.get("parsed_output")
    answer = (
        parsed.get("answer")
        if isinstance(parsed, dict)
        else None
    )

    passage_ids = payload.get("passage_ids")
    passage_count = (
        len(passage_ids)
        if isinstance(passage_ids, list)
        else 0
    )

    usage = (
        payload.get("provider_metadata", {})
        .get("usage", {})
    )

    retriever = payload.get("retriever")

    if payload.get("condition") == "WITHOUT_CONTEXT":
        condition_label = "WITHOUT_CONTEXT"
    else:
        condition_label = f"WITH_CONTEXT:{retriever}"

    records.append(
        {
            "path": str(path),
            "position": payload.get("position"),
            "sample_id": payload.get("sample_id"),
            "condition": payload.get("condition"),
            "condition_label": condition_label,
            "retriever": retriever,
            "logical_model": payload.get("logical_model_id"),
            "physical_model": payload.get("physical_model_id"),
            "status": payload.get("status"),
            "finish_reason": payload.get("finish_reason"),
            "selected_k": payload.get("selected_k"),
            "passage_count": passage_count,
            "elapsed_seconds": payload.get("elapsed_seconds"),
            "completion_tokens": usage.get("completion_tokens"),
            "answer_chars": len(answer) if isinstance(answer, str) else None,
        }
    )

generation = pd.DataFrame(records)

print("generation rows:", len(generation))
generation.head()

In [ ]:
# Exact matrix completeness check.

expected_conditions = {
    "WITHOUT_CONTEXT",
    "WITH_CONTEXT:dpr",
    "WITH_CONTEXT:bm25",
    "WITH_CONTEXT:contriever",
}

observed_conditions = set(generation["condition_label"])

assert observed_conditions == expected_conditions, (
    observed_conditions
)

models = sorted(generation["logical_model"].unique())

assert len(models) == 3, models

counts = (
    generation
    .groupby(["condition_label", "logical_model"])
    .size()
    .unstack(fill_value=0)
)

assert (counts == 948).all().all(), counts
assert len(generation) == 11_376, len(generation)

print("PASS: exact ASQA Sprint-1 matrix = 11,376 generations")
counts

### Generation Status Handling

Frozen status policy:

- `OK`: eligible for normal correctness scoring
- `REFUSAL`: correctness = 0 where defined
- `PARSE_FAILURE`: correctness = NA
- `TRUNCATED`: correctness = NA
- `ERROR`: correctness = NA after governed infrastructure retries

Valid content outcomes are not regenerated merely to improve the result.

In [ ]:
status_table = (
    generation
    .groupby(
        ["condition_label", "logical_model", "status"]
    )
    .size()
    .unstack(fill_value=0)
)

status_table

In [ ]:
status_totals = (
    generation["status"]
    .value_counts()
    .rename_axis("status")
    .to_frame("count")
)

assert len(generation) == status_totals["count"].sum()

status_totals

## Context Integrity

In [ ]:
with_context = generation[
    generation["condition"] == "WITH_CONTEXT"
].copy()

without_context = generation[
    generation["condition"] == "WITHOUT_CONTEXT"
].copy()

assert len(with_context) == 948 * 3 * 3
assert len(without_context) == 948 * 3

assert (with_context["selected_k"] == 5).all()
assert (with_context["passage_count"] == 5).all()

assert without_context["selected_k"].isna().all()
assert (without_context["passage_count"] == 0).all()

print("PASS: every WITH_CONTEXT generation uses exactly 5 passages")
print("PASS: WITHOUT_CONTEXT contains no passage IDs")

## ASQA Answer Correctness

The frozen primary correctness metric is:

**Official ASQA Disambig-F1 / QA-F1**

For each official disambiguated QA aspect, the generated long-form answer is
used as the QA evaluator context. The extracted answer is compared against
the official short-answer aliases using normalized token F1, then scores are
averaged across aspects and finally across original ASQA questions.

Important:

- retrieved passages are not supplied to the QA evaluator
- retrieval metadata is not supplied to the QA evaluator
- gold contexts are not supplied to the QA evaluator
- equal original-question weighting is preserved

The official scorer implementation, QA-model snapshot, tokenizer,
Transformers environment, windowing behavior, and provenance still need to
be physically frozen and parity-validated.

Therefore this notebook intentionally does **not** substitute generic
Token-F1, ROUGE, alias matching, or an LLM judge for QA-F1.

In [ ]:
# Scorable population under the frozen status policy.
#
# This is NOT QA-F1. It only reports how many generations are eligible
# for correctness evaluation once the official scorer is installed.

generation["correctness_measurable"] = generation["status"].isin(
    {"OK", "REFUSAL"}
)

measurable = (
    generation
    .groupby(["condition_label", "logical_model"])
    ["correctness_measurable"]
    .agg(["sum", "count"])
)

measurable["measurable_rate"] = (
    measurable["sum"] / measurable["count"]
)

measurable

## Does Retrieved Context Actually Help?

**Pending official ASQA QA-F1 execution.**

No answer-quality conclusion is drawn from generation status, answer length,
latency, or lexical overlap.

When the official scorer is implementation-validated, this section will
compare each WITH_CONTEXT retriever against the matching WITHOUT_CONTEXT
generation for the same question and LLM using the frozen paired statistical
protocol.

## Retrieval Evaluation

ASQA retrieval effectiveness is scientifically separate from answer
correctness.

The frozen ASQA retrieval layer defines:

- SRecall@5
- alpha-nDCG@5
- corpus coverability
- c* diagnostics

These require the frozen ASQA aspect-to-passage matching instrument and its
provenance. They are not approximated in this baseline notebook.

The three completed Sprint-1 retrieval systems currently provide the Top-5
passages used for generation. ColBERT will be added when available.

## Metric Coverage in This Baseline

| Metric / check | Current state |
|---|---|
| Generation completeness | Complete |
| Generation status integrity | Complete |
| Exact five-passage context validation | Complete |
| Official ASQA QA-F1 / Disambig-F1 | Pending scorer implementation |
| Answer-side alias coverage | Pending implementation |
| SRecall@5 | Pending retrieval matcher execution |
| alpha-nDCG@5 | Pending retrieval matcher execution |
| Faithfulness | Later Sprint-3 evaluation |
| ColBERT Sprint-1 condition | Pending |

## Main Findings

At this stage the notebook establishes the completed experimental matrix,
not the final answer-quality result.

Verified:

- 948 protected ASQA questions
- three completed relevance retrievers
- three frozen LLMs
- WITHOUT_CONTEXT baseline
- 11,376 generation artifacts
- exactly five passages for every WITH_CONTEXT generation
- frozen status handling
- no correctness approximation substituted for official QA-F1

The substantive question — whether retrieved context improves ASQA answer
correctness — will be answered after the official ASQA scorer implementation
is frozen and executed.

## Reproducibility Audit

In [ ]:
audit = pd.DataFrame(
    [
        ["Generation root exists", GEN_ROOT.is_dir()],
        ["Generation rows = 11,376", len(generation) == 11_376],
        [
            "Three logical LLM conditions",
            generation["logical_model"].nunique() == 3,
        ],
        [
            "Three completed retrievers",
            set(
                generation.loc[
                    generation["condition"] == "WITH_CONTEXT",
                    "retriever",
                ].dropna()
            )
            == {"dpr", "bm25", "contriever"},
        ],
        [
            "All WITH_CONTEXT rows have five passages",
            bool((with_context["passage_count"] == 5).all()),
        ],
        [
            "All WITH_CONTEXT rows have selected_k=5",
            bool((with_context["selected_k"] == 5).all()),
        ],
        [
            "No WITHOUT_CONTEXT passage IDs",
            bool((without_context["passage_count"] == 0).all()),
        ],
    ],
    columns=["Check", "PASS"],
)

assert audit["PASS"].all(), audit[~audit["PASS"]]

audit

## Limitations and Next Steps

1. Add ColBERT to this same notebook after its full-corpus retrieval and
   Sprint-1 generation complete.
2. Freeze and parity-test the official ASQA QA-F1 evaluator implementation.
3. Compute status-aware QA-F1 on the protected matrix without changing any
   scoring rule based on observed outcomes.
4. Add the frozen ASQA retrieval metrics when their matcher implementation
   is ready.
5. Do not rerun TRUNCATED or PARSE_FAILURE generations merely to improve
   reported performance.

## Reproducibility

Primary protocol references:

- `docs/sprint3/GENERATION_PROTOCOL.md`
- `docs/sprint3/ANSWER_CORRECTNESS_PROTOCOL.md`
- `docs/sprint3/ASQA_RETRIEVAL_METRIC_PROTOCOL.md`
- `docs/sprint3/ASQA_SUPERVISOR_CLARIFICATION_2026-09-03.md`
- `docs/sprint3/PROJECT_HANDOFF_2026-09-09.md`

Generation artifacts remain outside Git. The notebook, protocols, manifests,
and implementation code are version-controlled.